In [1]:
from pathlib import Path

import pandas as pd

# March 2026 release exploration

This notebook is a **structure-first pass** over the March 2026 Primary Care Dementia Data release, following the same general approach as the June 2026 exploration.

The March release is noticeably more fragmented than June:

- measures are spread across several separate CSV files rather than consolidated into a single Sub-ICB file and a single practice file
- most analytical files contain a **13-month time series**, from March 2025 to March 2026
- some files report Sub-ICB data only, while others repeat the same measures across Sub-ICB, ICB, region and country levels
- the release also contains supporting files for practice mappings, latest practice submission dates, a data dictionary and an Excel summary workbook

The aim here is **not yet to analyse dementia outcomes**. It is to understand what each file contains, what one row represents, how the files differ, and what would need to be transformed later for the Atlas.

> The supplied data dictionary also contains definitions for `pcdem-prac-ass-plans` and `pcdem-prac-anti-psy`. Matching source files are not present in this local March source set, so they are not analysed in this notebook.

## 1. Load the March release

In [2]:
MARCH_RAW_DIR = Path("../data/raw/pcdd/2026-03")

march_paths = {
    "nhs_rate": MARCH_RAW_DIR / "pcdem-nhs-rate-mar-2026.csv",
    "la_rate": MARCH_RAW_DIR / "pcdem-la-rate-mar-2026.csv",
    "sicbl_age_sex": MARCH_RAW_DIR / "pcdem-sicbl-age-sex-mar-2026.csv",
    "sicbl_ethnicity": MARCH_RAW_DIR / "pcdem-sicbl-ethnicity-mar-2026.csv",
    "sicbl_dem_type": MARCH_RAW_DIR / "pcdem-sicbl-dem-type-mar-2026.csv",
    "sicbl_res_type": MARCH_RAW_DIR / "pcdem-sicbl-res-type-mar-2026.csv",
    "sicbl_incidence_onset_delirium": MARCH_RAW_DIR / "pcdem-sicbl-incidence-onset-delirium-mar-2026.csv",
    "sicbl_comor_pall": MARCH_RAW_DIR / "pcdem-sicbl-comor-pall-care-mar-2026.csv",
    "sicbl_cog_imp": MARCH_RAW_DIR / "pcdem_sicbl-cog-imp-mar-2026.csv",
    "practice_data_date": MARCH_RAW_DIR / "pcdem-prac-data-date-mar-2026.csv",
    "mapping": MARCH_RAW_DIR / "gp-reg-pat-prac-map-03-2026.csv",
    "summary": MARCH_RAW_DIR / "pcdem-sum-mar-2026.xlsx",
    "dictionary": MARCH_RAW_DIR / "PCDD-2526-data-dictionary.xlsx",
}

nhs_rate = pd.read_csv(march_paths["nhs_rate"])
la_rate = pd.read_csv(march_paths["la_rate"])
sicbl_age_sex = pd.read_csv(march_paths["sicbl_age_sex"])
sicbl_ethnicity = pd.read_csv(march_paths["sicbl_ethnicity"])
sicbl_dem_type = pd.read_csv(march_paths["sicbl_dem_type"])
sicbl_res_type = pd.read_csv(march_paths["sicbl_res_type"])
sicbl_incidence_onset_delirium = pd.read_csv(
    march_paths["sicbl_incidence_onset_delirium"]
)
sicbl_comor_pall = pd.read_csv(march_paths["sicbl_comor_pall"])
sicbl_cog_imp = pd.read_csv(march_paths["sicbl_cog_imp"])
practice_data_date = pd.read_csv(march_paths["practice_data_date"])
mapping = pd.read_csv(march_paths["mapping"])

datasets = {
    "nhs_rate": nhs_rate,
    "la_rate": la_rate,
    "sicbl_age_sex": sicbl_age_sex,
    "sicbl_ethnicity": sicbl_ethnicity,
    "sicbl_dem_type": sicbl_dem_type,
    "sicbl_res_type": sicbl_res_type,
    "sicbl_incidence_onset_delirium": sicbl_incidence_onset_delirium,
    "sicbl_comor_pall": sicbl_comor_pall,
    "sicbl_cog_imp": sicbl_cog_imp,
    "practice_data_date": practice_data_date,
    "mapping": mapping,
}

for name, dataset in datasets.items():
    print(f"Displaying dataset: {name}")
    display(dataset.head())
    print(f"Shape of {name}: {dataset.shape}")
    print()

Displaying dataset: nhs_rate


,INDICATOR,ORG_TYPE,ORG_CODE,ONS_CODE,NAME,ACH_DATE,MEASURE,VALUE,DQ
0,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_RESPONSIBILITY,ENG,ENG,ENGLAND,31-Mar-26,DEMENTIA_ESTIMATE_65_PLUS,751551.4,NaN
1,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_RESPONSIBILITY,ENG,ENG,ENGLAND,31-Mar-26,DEMENTIA_REGISTER_65_PLUS,498122.0,NaN
2,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_RESPONSIBILITY,ENG,ENG,ENGLAND,31-Mar-26,DIAG_RATE_65_PLUS,66.3,NaN
3,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_RESPONSIBILITY,ENG,ENG,ENGLAND,31-Mar-26,DIAG_RATE_65_PLUS_LL,59.7,NaN
4,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_RESPONSIBILITY,ENG,ENG,ENGLAND,31-Mar-26,DIAG_RATE_65_PLUS_UL,71.8,NaN


Shape of nhs_rate: (10140, 9)

Displaying dataset: la_rate


,INDICATOR,ORG_TYPE,ONS_CODE,NAME,ACH_DATE,MEASURE,VALUE,DQ
0,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_GEOGRAPHICAL,E92000001,ENGLAND,31-Mar-26,DEMENTIA_ESTIMATE_65_PLUS,751551.4,NaN
1,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_GEOGRAPHICAL,E92000001,ENGLAND,31-Mar-26,DEMENTIA_REGISTER_65_PLUS,498122.0,NaN
2,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_GEOGRAPHICAL,E92000001,ENGLAND,31-Mar-26,DIAG_RATE_65_PLUS,66.3,NaN
3,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_GEOGRAPHICAL,E92000001,ENGLAND,31-Mar-26,DIAG_RATE_65_PLUS_LL,59.7,NaN
4,DEMENTIA: 65+ ESTIMATED DIAGNOSIS RATE,COUNTRY_GEOGRAPHICAL,E92000001,ENGLAND,31-Mar-26,DIAG_RATE_65_PLUS_UL,71.8,NaN


Shape of la_rate: (29415, 8)

Displaying dataset: sicbl_age_sex


...

Shape of sicbl_cog_imp: (44616, 7)

Displaying dataset: practice_data_date


,REGION_ODS_CODE,REGION_ONS_CODE,REGION_NAME,ICB_ODS_CODE,ICB_ONS_CODE,ICB_NAME,SUB_ICB_ODS_CODE,SUB_ICB_ONS_CODE,SUB_ICB_NAME,PCN_ODS_CODE,PCN_NAME,PRACTICE_CODE,PRACTICE_NAME,LATEST_DATA_SUBMISSION
0,Y56,E40000003,London,QKK,E54000030,NHS South East London Integrated Care Board,72Q,E38000244,NHS South East London ICB - 72Q,U00254,BLACKHEATH AND CHARLTON PCN,G83001,MANOR BROOK PMS,2026-03-31
1,Y56,E40000003,London,QKK,E54000030,NHS South East London Integrated Care Board,72Q,E38000244,NHS South East London ICB - 72Q,U88820,CLOCKTOWER PCN,G83002,THE WESTWOOD SURGERY,2026-03-31
2,Y56,E40000003,London,QKK,E54000030,NHS South East London Integrated Care Board,72Q,E38000244,NHS South East London ICB - 72Q,U50197,FROGNAL PCN,G83004,BARNARD MEDICAL GROUP,2026-03-31
3,Y56,E40000003,London,QKK,E54000030,NHS South East London Integrated Care Board,72Q,E38000244,NHS South East London ICB - 72Q,U07140,APL BEXLEY PCN,G83006,THE ALBION SURGERY,2026-03-31
4,Y56,E40000003,London,QKK,E54000030,NHS South East London Integrated Care Board,72Q,E38000244,NHS South East London ICB - 72Q,U88820,CLOCKTOWER PCN,G83009,BELLEGROVE SURGERY,2026-03-31


Shape of practice_data_date: (6124, 14)

Displaying dataset: mapping


,PUBLICATION,EXTRACT_DATE,PRACTICE_CODE,PRACTICE_NAME,PRACTICE_POSTCODE,PCN_CODE,PCN_NAME,ONS_SUB_ICB_LOCATION_CODE,SUB_ICB_LOCATION_CODE,SUB_ICB_LOCATION_NAME,ONS_ICB_CODE,ICB_CODE,ICB_NAME,ONS_COMM_REGION_CODE,COMM_REGION_CODE,COMM_REGION_NAME,SUPPLIER_NAME
0,GP_PRAC_PAT_LIST,01Mar2026,A81001,THE DENSHAM SURGERY,TS18 1HU,U89141,STOCKTON PCN,E38000247,16C,NHS North East and North Cumbria ICB - 16C,E54000050,QHM,NHS North East and North Cumbria Integrated Ca...,E40000012,Y63,North East and Yorkshire,TPP
1,GP_PRAC_PAT_LIST,01Mar2026,A81002,QUEENS PARK MEDICAL CENTRE,TS18 2AW,U07032,NORTH STOCKTON PCN,E38000247,16C,NHS North East and North Cumbria ICB - 16C,E54000050,QHM,NHS North East and North Cumbria Integrated Ca...,E40000012,Y63,North East and Yorkshire,TPP
2,GP_PRAC_PAT_LIST,01Mar2026,A81004,ACKLAM MEDICAL CENTRE,TS5 8SB,U02671,GREATER MIDDLESBROUGH PCN,E38000247,16C,NHS North East and North Cumbria ICB - 16C,E54000050,QHM,NHS North East and North Cumbria Integrated Ca...,E40000012,Y63,North East and Yorkshire,TPP
3,GP_PRAC_PAT_LIST,01Mar2026,A81005,SPRINGWOOD SURGERY,TS14 7DJ,U07842,EAST CLEVELAND PCN,E38000247,16C,NHS North East and North Cumbria ICB - 16C,E54000050,QHM,NHS North East and North Cumbria Integrated Ca...,E40000012,Y63,North East and Yorkshire,TPP
4,GP_PRAC_PAT_LIST,01Mar2026,A81006,TENNANT STREET MEDICAL PRACTICE,TS18 2AT,U07032,NORTH STOCKTON PCN,E38000247,16C,NHS North East and North Cumbria ICB - 16C,E54000050,QHM,NHS North East and North Cumbria Integrated Ca...,E40000012,Y63,North East and Yorkshire,TPP


Shape of mapping: (6169, 17)



In [3]:
dictionary_xls = pd.ExcelFile(march_paths["dictionary"])
summary_xls = pd.ExcelFile(march_paths["summary"])

print("Data dictionary sheets:")
display(dictionary_xls.sheet_names)

print("Summary workbook sheets:")
display(summary_xls.sheet_names)

Data dictionary sheets:


['Title Sheet',
 'pcdem-nhs-rate',
 'pcdem-la-rate',
 'pcdem-prac-ass-plans',
 'pcdem-sicbl-ethnicity',
 'pcdem-sicbl-dem-type',
 'pcdem-sicbl-res-type',
 'pcdem-prac-anti-psy',
 'pcdem-sicbl-age-sex',
 'pcdem-sicbl-comor-pall-care',
 'pcdem_sicbl-incidence-onset-del',
 'pcdem_sicbl-cog-imp',
 'pcdem-prac-map-mmm-yyyy',
 'pcdem-prac-data-date-mmm-yyyy']

Summary workbook sheets:


['Title sheet',
 'Notes and definitions',
 'Table 1',
 'Table 2',
 'Table 3a',
 'Table 3b',
 'Table 3c',
 'Table 4',
 'Table 5']

## 2. Dataset structure and grain

Before analysing values, establish:

- what one row represents in each file
- which geography or organisation level is present
- whether the file contains a single snapshot or a time series
- which columns identify the measure
- whether every reporting entity contributes the same number of records

In [4]:
structure = []

for name, df in datasets.items():
    date_count = df["ACH_DATE"].nunique() if "ACH_DATE" in df.columns else None

    structure.append(
        {
            "DATASET": name,
            "ROWS": len(df),
            "COLUMNS": len(df.columns),
            "REPORTING_DATES": date_count,
        }
    )

pd.DataFrame(structure)

,DATASET,ROWS,COLUMNS,REPORTING_DATES
0,nhs_rate,10140,9,13.0
1,la_rate,29415,8,13.0
2,sicbl_age_sex,24804,12,13.0
3,sicbl_ethnicity,11024,12,13.0
4,sicbl_dem_type,10812,12,13.0
5,sicbl_res_type,8268,12,13.0
6,sicbl_incidence_onset_delirium,9984,7,13.0
7,sicbl_comor_pall,6084,7,13.0
8,sicbl_cog_imp,44616,7,13.0
9,practice_data_date,6124,14,NaN


### Reporting-period coverage

A major difference from the June 2026 release is visible immediately.

Most March analytical files contain **13 reporting dates**. They carry a rolling time series ending in March 2026, rather than only the latest reporting quarter.

The mapping and latest-submission-date files are supporting snapshots and therefore have a different structure.

In [26]:
time_series_datasets = {
    name: df
    for name, df in datasets.items()
    if "ACH_DATE" in df.columns
}

date_coverage = []

for name, df in time_series_datasets.items():
    dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True, format='mixed')

    date_coverage.append(
        {
            "DATASET": name,
            "PERIODS": dates.nunique(),
            "FIRST_DATE": dates.min().date(),
            "LAST_DATE": dates.max().date(),
        }
    )

pd.DataFrame(date_coverage)

,DATASET,PERIODS,FIRST_DATE,LAST_DATE
0,nhs_rate,13,2025-03-31,2026-03-31
1,la_rate,13,2025-03-31,2026-03-31
2,sicbl_age_sex,13,2025-03-31,2026-03-31
3,sicbl_ethnicity,13,2025-03-31,2026-03-31
4,sicbl_dem_type,13,2025-03-31,2026-03-31
5,sicbl_res_type,13,2025-03-31,2026-03-31
6,sicbl_incidence_onset_delirium,13,2025-03-31,2026-03-31
7,sicbl_comor_pall,13,2025-03-31,2026-03-31
8,sicbl_cog_imp,13,2025-03-31,2026-03-31


## NHS Rate Dataset

**Publisher definition:** Dementia diagnosis rate from 65+ by NHS organisation

**Shape:** 10,140 rows × 9 columns

### What does one row represent?

One row reports one dementia diagnosis-rate measure for an NHS reporting entity at a reporting-period end date.

The effective grain is:

`NHS reporting entity + reporting date + measure = one value`

Key fields:

- `INDICATOR` - diagnosis-rate indicator identifier
- `ORG_TYPE` - organisation/geographic level
- `ORG_CODE` - Organisation Data Service identifier
- `ONS_CODE` - ONS geography code
- `NAME` - organisation name
- `ACH_DATE` - reporting-period end date
- `MEASURE` - identifies what the value represents
- `VALUE` - value associated with the measure
- `DQ` - data-quality flag

The same five diagnosis-rate measures used in the June release are present here:

- `DEMENTIA_ESTIMATE_65_PLUS`
- `DEMENTIA_REGISTER_65_PLUS`
- `DIAG_RATE_65_PLUS`
- `DIAG_RATE_65_PLUS_LL`
- `DIAG_RATE_65_PLUS_UL`

### Reporting dates

In [6]:
nhs_rate["ACH_DATE"].value_counts().sort_index().to_frame()

,count
ACH_DATE,
28-Feb-26,780
30-Apr-25,780
30-Jun-25,780
30-Nov-25,780
30-Sep-25,780
31-Aug-25,780
31-Dec-25,780
31-Jan-26,780
31-Jul-25,780


### ORG_TYPE

In [7]:
# Use the latest reporting period to inspect the March 2026 structure.
latest_nhs_date = pd.to_datetime(nhs_rate["ACH_DATE"], dayfirst=True).max()

nhs_rate_latest = nhs_rate[
    pd.to_datetime(nhs_rate["ACH_DATE"], dayfirst=True) == latest_nhs_date
]

display(nhs_rate_latest["ORG_TYPE"].value_counts().to_frame())

display(
    nhs_rate_latest.groupby("ORG_TYPE")["ORG_CODE"]
    .nunique()
    .to_frame(name="UNIQUE_ORG_COUNT")
)

/tmp/ipykernel_200148/3994892119.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  latest_nhs_date = pd.to_datetime(nhs_rate["ACH_DATE"], dayfirst=True).max()
/tmp/ipykernel_200148/3994892119.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(nhs_rate["ACH_DATE"], dayfirst=True) == latest_nhs_date


,count
ORG_TYPE,
SUB_ICB_LOC,530
ICB,210
NHS_REGION,35
COUNTRY_RESPONSIBILITY,5


,UNIQUE_ORG_COUNT
ORG_TYPE,
COUNTRY_RESPONSIBILITY,1
ICB,42
NHS_REGION,7
SUB_ICB_LOC,106


### MEASURE

In [8]:
display(nhs_rate_latest["MEASURE"].value_counts().to_frame())

nhs_rate_latest.groupby(
    ["ORG_TYPE", "ORG_CODE"]
)["MEASURE"].nunique().value_counts().to_frame(
    name="ENTITY_COUNT"
)

,count
MEASURE,
DEMENTIA_ESTIMATE_65_PLUS,156
DEMENTIA_REGISTER_65_PLUS,156
DIAG_RATE_65_PLUS,156
DIAG_RATE_65_PLUS_LL,156
DIAG_RATE_65_PLUS_UL,156


,ENTITY_COUNT
MEASURE,
5,156


### NHS Rate Dataset Structure Breakdown

The full file contains **10,140 observations across 13 monthly reporting periods** from March 2025 to March 2026.

For the latest period, 31 March 2026, there are **780 rows across 156 reporting entities**:

| Organisation Type (`ORG_TYPE`) | Rows | Unique Reporting Entities |
| :--- | ---: | ---: |
| `SUB_ICB_LOC` | 530 | 106 |
| `ICB` | 210 | 42 |
| `NHS_REGION` | 35 | 7 |
| `COUNTRY_RESPONSIBILITY` | 5 | 1 |
| **Total** | **780** | **156** |

Every reporting entity has the same five rate measures in the March 2026 period.

The latest-period grain is therefore:

`reporting entity + reporting date + one of 5 rate measures = one value`

## Local Authority Rate Dataset

**Publisher definition:** Dementia diagnosis rate from 65+ by Local Authority organisation

**Shape:** 29,415 rows × 8 columns

### What does one row represent?

This file follows the same diagnosis-rate pattern as the NHS rate file, but uses local-government geographies rather than NHS commissioning organisations.

The effective grain is:

`local-authority geography level + ONS geography code + reporting date + measure = one value`

`ORG_TYPE` matters here because the same ONS geography can participate in more than one reporting level.

The file includes:

- lower-tier local authorities (`LTLA`)
- upper-tier local authorities (`UTLA`)
- Government Office Regions (`GOR`)
- England (`COUNTRY_GEOGRAPHICAL`)

In [9]:
latest_la_date = pd.to_datetime(la_rate["ACH_DATE"], dayfirst=True).max()

la_rate_latest = la_rate[
    pd.to_datetime(la_rate["ACH_DATE"], dayfirst=True) == latest_la_date
]

display(la_rate_latest["ORG_TYPE"].value_counts().to_frame())

display(
    la_rate_latest.groupby("ORG_TYPE")["ONS_CODE"]
    .nunique()
    .to_frame(name="UNIQUE_GEOGRAPHY_COUNT")
)

display(la_rate_latest["MEASURE"].value_counts().to_frame())

/tmp/ipykernel_200148/894384214.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  latest_la_date = pd.to_datetime(la_rate["ACH_DATE"], dayfirst=True).max()
/tmp/ipykernel_200148/894384214.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(la_rate["ACH_DATE"], dayfirst=True) == latest_la_date


,count
ORG_TYPE,
LTLA,1480
UTLA,765
GOR,45
COUNTRY_GEOGRAPHICAL,5


,UNIQUE_GEOGRAPHY_COUNT
ORG_TYPE,
COUNTRY_GEOGRAPHICAL,1
GOR,9
LTLA,296
UTLA,153


,count
MEASURE,
DEMENTIA_ESTIMATE_65_PLUS,459
DEMENTIA_REGISTER_65_PLUS,459
DIAG_RATE_65_PLUS,459
DIAG_RATE_65_PLUS_LL,459
DIAG_RATE_65_PLUS_UL,459


In [10]:
# Confirm that each March 2026 geography has all five diagnosis-rate measures.
display(
    la_rate_latest.groupby(
        ["ORG_TYPE", "ONS_CODE"]
    )["MEASURE"].nunique().value_counts().to_frame(
        name="GEOGRAPHY_COUNT"
    )
)

print("DQ values in March 2026:")
display(la_rate_latest["DQ"].value_counts(dropna=False).to_frame())

,GEOGRAPHY_COUNT
MEASURE,
5,459


DQ values in March 2026:


,count
DQ,
NaN,2275
1.0,20


### Local Authority Rate Dataset Structure Breakdown

The March 2026 period contains **2,295 rows across 459 geography-level entities**:

| Organisation Type | Rows | Unique geography-level entities |
| :--- | ---: | ---: |
| `LTLA` | 1,480 | 296 |
| `UTLA` | 765 | 153 |
| `GOR` | 45 | 9 |
| `COUNTRY_GEOGRAPHICAL` | 5 | 1 |
| **Total** | **2,295** | **459** |

Each geography-level entity has the same five diagnosis-rate measures in March 2026.

Unlike the NHS rate file, the number of rows is not identical across all 13 reporting months. This is worth retaining rather than assuming that every historical month has the same geography coverage.

There are also March 2026 rows with `DQ = 1`, so the data-quality flag should be preserved in later processing.

## Split Sub-ICB Demographic Datasets

In the March publication, several dementia-register breakdowns are held in **separate files**.

These four files are Sub-ICB level only:

- age and sex
- ethnicity
- dementia type
- residential type

They use a wider 12-column schema which carries the Region, ICB and Sub-ICB identifiers on each row.

Importantly, the category is encoded directly in the `Measure` value. This is different from the June consolidated Sub-ICB file, where fields such as `BREAKDOWN`, `AGE`, `GENDER`, `ETHNICITY`, `DEMENTIA_TYPE` and `RESIDENTIAL_TYPE` make those dimensions explicit.

In [11]:
sub_icb_breakdown_datasets = {
    "age_sex": sicbl_age_sex,
    "ethnicity": sicbl_ethnicity,
    "dementia_type": sicbl_dem_type,
    "residential_type": sicbl_res_type,
}

summary_rows = []

for name, df in sub_icb_breakdown_datasets.items():
    dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
    latest_date = dates.max()
    latest = df[dates == latest_date]

    summary_rows.append(
        {
            "DATASET": name,
            "TOTAL_ROWS": len(df),
            "REPORTING_PERIODS": dates.nunique(),
            "LATEST_ROWS": len(latest),
            "SUB_ICBS": latest["SUB_ICB_ODS_CODE"].nunique(),
            "MEASURES": latest["Measure"].nunique(),
        }
    )

pd.DataFrame(summary_rows)

/tmp/ipykernel_200148/145691877.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/145691877.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/145691877.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/145691877.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure pars

,DATASET,TOTAL_ROWS,REPORTING_PERIODS,LATEST_ROWS,SUB_ICBS,MEASURES
0,age_sex,24804,13,1908,106,18
1,ethnicity,11024,13,848,106,8
2,dementia_type,10812,13,848,106,8
3,residential_type,8268,13,636,106,6


### Age and Sex

**Shape:** 24,804 rows × 12 columns

### What does one row represent?

One row reports the count for one dementia-register age/sex category for one Sub-ICB at one reporting date.

The effective grain is:

`Sub-ICB + reporting date + age/sex measure = one value`

The `Measure` field contains the category itself, for example:

- `FEMALE_AGED_65_69`
- `MALE_AGED_80_84`
- `ALL_AGED_90_PLUS`

In [12]:
latest_age_sex_date = pd.to_datetime(
    sicbl_age_sex["ACH_DATE"], dayfirst=True
).max()

age_sex_latest = sicbl_age_sex[
    pd.to_datetime(sicbl_age_sex["ACH_DATE"], dayfirst=True)
    == latest_age_sex_date
]

display(age_sex_latest["Measure"].value_counts().to_frame())

display(
    age_sex_latest.groupby("SUB_ICB_ODS_CODE").size()
    .value_counts()
    .sort_index()
    .to_frame(name="SUB_ICB_COUNT")
)

/tmp/ipykernel_200148/2718889507.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  latest_age_sex_date = pd.to_datetime(
/tmp/ipykernel_200148/2718889507.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(sicbl_age_sex["ACH_DATE"], dayfirst=True)


,count
Measure,
ALL_AGED_65_69,106
ALL_AGED_70_74,106
ALL_AGED_75_79,106
ALL_AGED_80_84,106
ALL_AGED_85_89,106
ALL_AGED_90_PLUS,106
FEMALE_AGED_65_69,106
FEMALE_AGED_70_74,106
FEMALE_AGED_75_79,106


,SUB_ICB_COUNT
18,106


### Age and Sex Structure Breakdown

For 31 March 2026:

- **106** Sub-ICB locations are present
- each Sub-ICB contributes exactly **18 rows**
- the 18 measures represent 6 age bands × female, male and all-sex totals

This gives **1,908 rows** in the latest period.

The file contains all 13 monthly reporting periods, giving **24,804 rows** overall.

### Ethnicity, Dementia Type and Residential Type

These three files use the same general grain:

`Sub-ICB + reporting date + category measure = one value`

The category is again embedded in the `Measure` field rather than stored in a dedicated category column.

In [13]:
for name, df in {
    "ETHNICITY": sicbl_ethnicity,
    "DEMENTIA TYPE": sicbl_dem_type,
    "RESIDENTIAL TYPE": sicbl_res_type,
}.items():
    dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
    latest = df[dates == dates.max()]

    print(f"\n{name}")
    print(f"Latest-period rows: {len(latest):,}")
    print(f"Sub-ICBs: {latest['SUB_ICB_ODS_CODE'].nunique():,}")
    print(f"Measures: {latest['Measure'].nunique():,}")
    display(latest["Measure"].value_counts().to_frame())

/tmp/ipykernel_200148/2610372731.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)



ETHNICITY
Latest-period rows: 848
Sub-ICBs: 106
Measures: 8


,count
Measure,
ASIAN_OR_ASIAN_BRITISH,106
BLACK_OR_AFRICAN_OR_CARIBBEAN_OR_BLACK_BRITISH,106
INCONCLUSIVE_ETHNIC_GROUP,106
MIXED_OR_MULTIPLE_ETHNIC_GROUPS,106
NOT_DEFINED,106
NOT_STATED,106
OTHER_ETHNIC_GROUP,106
WHITE,106



DEMENTIA TYPE
Latest-period rows: 848
Sub-ICBs: 106
Measures: 8


/tmp/ipykernel_200148/2610372731.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)


,count
Measure,
ALZHEIMERS_DISEASE,106
FRONTOTEMPORAL,106
INCONCLUSIVE_DEMENTIA_TYPE,106
LEWY_BODY,106
MIXED_DEMENTIA_TYPES,106
OTHER_SPECIFIED_DEMENTIA_TYPES,106
OTHER_UNSPECIFIED_DEMENTIA_TYPES,106
VASCULAR_DEMENTIA,106



RESIDENTIAL TYPE
Latest-period rows: 636
Sub-ICBs: 106
Measures: 6


/tmp/ipykernel_200148/2610372731.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)


,count
Measure,
INCONCLUSIVE_RESIDENTIAL_TYPE,106
NO_PERMANENT_ADDRESS,106
NURSING_HOME,106
OTHER_RESIDENTIAL_TYPE,106
PRIVATE_RESIDENCE,106
RESIDENTIAL_CARE_HOME,106


### Demographic Split Files Structure Breakdown

For 31 March 2026:

| File | Measures per Sub-ICB | Latest rows |
| :--- | ---: | ---: |
| Age / sex | 18 | 1,908 |
| Ethnicity | 8 | 848 |
| Dementia type | 8 | 848 |
| Residential type | 6 | 636 |

All four files contain **106 Sub-ICBs** in the latest period.

One small historical difference is visible inside the dementia-type file: the earliest month in the 13-month series has fewer rows than later months. This is a useful warning not to assume that the measure set is completely static across the whole historical series.

In [14]:
# Check row counts by reporting month for the four Sub-ICB breakdown files.
monthly_rows = {}

for name, df in sub_icb_breakdown_datasets.items():
    dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)

    monthly_rows[name] = (
        pd.DataFrame({"DATE": dates})
        .value_counts("DATE")
        .sort_index()
    )

pd.DataFrame(monthly_rows)

/tmp/ipykernel_200148/2329964612.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/2329964612.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/2329964612.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/2329964612.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure pars

,age_sex,ethnicity,dementia_type,residential_type
DATE,,,,
2025-03-31,1908,848,636,636
2025-04-30,1908,848,848,636
2025-05-31,1908,848,848,636
2025-06-30,1908,848,848,636
2025-07-31,1908,848,848,636
2025-08-31,1908,848,848,636
2025-09-30,1908,848,848,636
2025-10-31,1908,848,848,636
2025-11-30,1908,848,848,636


## NHS Organisation Measure Files

Three other March files have a different structure:

- young onset, incidence and delirium
- comorbidities and palliative care
- cognitive impairment

Despite the historical filenames containing `sicbl`, these files are **not Sub-ICB-only**. They report values at several NHS organisation levels:

`Sub-ICB → ICB → Region → Country`

Their effective grain is:

`NHS reporting entity + reporting date + measure = one value`

In [15]:
nhs_measure_datasets = {
    "incidence_onset_delirium": sicbl_incidence_onset_delirium,
    "comorbidities_palliative_care": sicbl_comor_pall,
    "cognitive_impairment": sicbl_cog_imp,
}

for name, df in nhs_measure_datasets.items():
    dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
    latest = df[dates == dates.max()]

    print(f"\n{name.upper()}")
    display(latest["ORG_TYPE"].value_counts().to_frame())
    display(latest["Measure"].value_counts().to_frame())


INCIDENCE_ONSET_DELIRIUM


/tmp/ipykernel_200148/341106333.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)


,count
ORG_TYPE,
SUB_ICB,530
ICB,210
REGION,35
COUNTRY,5


,count
Measure,
INCIDENCE,156
DELIRIUM_12M,156
YOUNG_ONSET,156
PAT_LIST_ALL,156
DEMENTIA_REGISTER,156



COMORBIDITIES_PALLIATIVE_CARE


...

,count
Measure,
PALLIATIVE_CARE,156
COMORBIDITIES,156
DEMENTIA_REGISTER_65_PLUS,156



COGNITIVE_IMPAIRMENT


/tmp/ipykernel_200148/341106333.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)


,count
ORG_TYPE,
SUB_ICB,2332
ICB,924
REGION,154
COUNTRY,22


,count
Measure,
MCI_FEMALE_AGED_45_49,156
MCI_FEMALE_AGED_40_44,156
MCI_FEMALE_AGED_50_54,156
MCI_FEMALE_AGED_55_59,156
MCI_FEMALE_AGED_85_89,156
MCI_MALE_AGED_60_64,156
MCI_MALE_AGED_55_59,156
MCI_MALE_AGED_50_54,156
MCI_MALE_AGED_45_49,156


### Young Onset, Incidence and Delirium

**Shape:** 9,984 rows × 7 columns

For March 2026, each of the **156 NHS reporting entities** has five records:

- `DEMENTIA_REGISTER`
- `PAT_LIST_ALL`
- `INCIDENCE`
- `YOUNG_ONSET`
- `DELIRIUM_12M`

The March 2025 period contains fewer rows than the later periods, so the five-measure structure should not be assumed across the complete 13-month series without checking the date.

### Comorbidities and Palliative Care

**Shape:** 6,084 rows × 7 columns

For March 2026, each of the **156 NHS reporting entities** has three records:

- `DEMENTIA_REGISTER_65_PLUS`
- `COMORBIDITIES`
- `PALLIATIVE_CARE`

The latest-period structure is regular: 156 entities × 3 measures = **468 rows**.

### Cognitive Impairment

**Shape:** 44,616 rows × 7 columns

For March 2026, each of the **156 NHS reporting entities** has **22 MCI age/sex measures**, producing **3,432 rows** in the latest period.

The measures cover female and male patients from ages 40-44 through to 90+.

In [16]:
# Compare monthly row counts and expose any structural changes inside the 13-month series.
monthly_nhs_measure_rows = {}

for name, df in nhs_measure_datasets.items():
    dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)

    monthly_nhs_measure_rows[name] = (
        pd.DataFrame({"DATE": dates})
        .value_counts("DATE")
        .sort_index()
    )

pd.DataFrame(monthly_nhs_measure_rows)

/tmp/ipykernel_200148/4240917966.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/4240917966.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)
/tmp/ipykernel_200148/4240917966.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(df["ACH_DATE"], dayfirst=True)


,incidence_onset_delirium,comorbidities_palliative_care,cognitive_impairment
DATE,,,
2025-03-31,624,468,3432
2025-04-30,780,468,3432
2025-05-31,780,468,3432
2025-06-30,780,468,3432
2025-07-31,780,468,3432
2025-08-31,780,468,3432
2025-09-30,780,468,3432
2025-10-31,780,468,3432
2025-11-30,780,468,3432


## Practice Latest Data Submission Dataset

**Publisher definition:** Latest extract date by GP practice

**Shape:** 6,124 rows × 14 columns

### What does one row represent?

One row represents one GP practice and the date on which dementia data were most recently received for that practice.

The effective grain is:

`GP practice = one row`

The file also carries the commissioning hierarchy for each practice:

`GP practice → PCN → Sub-ICB → ICB → NHS region`

In [17]:
print(f"Rows: {len(practice_data_date):,}")
print(
    f"Unique practice codes: "
    f"{practice_data_date['PRACTICE_CODE'].nunique():,}"
)

display(
    practice_data_date["LATEST_DATA_SUBMISSION"]
    .value_counts(dropna=False)
    .to_frame()
)

Rows: 6,124
Unique practice codes: 6,124


,count
LATEST_DATA_SUBMISSION,
2026-03-31,6115
2026-02-28,8
2025-09-30,1


### Practice Latest Data Submission Structure Breakdown

The file contains **6,124 rows and 6,124 unique GP practice codes**, confirming one row per practice.

Most practices have a latest submission date of **31 March 2026**, but a small number use an earlier submission date.

That makes this a useful supporting file when interpreting practice coverage: a practice can still be represented even when its latest available dementia extract is not from the current reporting month.

## Mapping Dataset

**Publisher definition:** Mapping of GP practices to PCNs, Sub-ICBs, ICBs and NHS commissioning regions

**Shape:** 6,169 rows × 17 columns

### What does one row represent?

The mapping dataset is a snapshot linking one GP practice to the organisational hierarchy above it.

The effective grain is:

`GP practice at mapping extract date = one row`

Compared with the June mapping file used in the other notebook, this March mapping also includes:

- `PCN_CODE`
- `PCN_NAME`
- `SUPPLIER_NAME`

In [18]:
print(f"Rows: {len(mapping):,}")
print(f"Unique practice codes: {mapping['PRACTICE_CODE'].nunique():,}")

display(mapping["EXTRACT_DATE"].value_counts().to_frame())

display(
    mapping[
        [
            "PCN_CODE",
            "SUB_ICB_LOCATION_CODE",
            "ICB_CODE",
            "COMM_REGION_CODE",
        ]
    ]
    .nunique()
    .to_frame(name="UNIQUE_COUNT")
)

Rows: 6,169
Unique practice codes: 6,169


,count
EXTRACT_DATE,
01Mar2026,6169


,UNIQUE_COUNT
PCN_CODE,1299
SUB_ICB_LOCATION_CODE,106
ICB_CODE,42
COMM_REGION_CODE,7


In [19]:
# Check whether any practice mappings are missing hierarchy fields.
mapping[
    [
        "PCN_CODE",
        "SUB_ICB_LOCATION_CODE",
        "ICB_CODE",
        "COMM_REGION_CODE",
    ]
].isna().sum().to_frame(name="MISSING_COUNT")

,MISSING_COUNT
PCN_CODE,0
SUB_ICB_LOCATION_CODE,0
ICB_CODE,0
COMM_REGION_CODE,0


### Practice Data-Date Comparison

In [20]:
# Compare practice coverage between the mapping snapshot and latest-submission file.
mapping_codes = set(mapping["PRACTICE_CODE"])
submission_codes = set(practice_data_date["PRACTICE_CODE"])

print(f"Mapping dataset codes: {len(mapping_codes):,}")
print(f"Latest-submission dataset codes: {len(submission_codes):,}")
print(f"Present in both: {len(mapping_codes & submission_codes):,}")
print(f"Mapping only: {len(mapping_codes - submission_codes):,}")
print(f"Latest-submission only: {len(submission_codes - mapping_codes):,}")

Mapping dataset codes: 6,169
Latest-submission dataset codes: 6,124
Present in both: 6,124
Mapping only: 45
Latest-submission only: 0


### Mapping Dataset Structure Breakdown

The mapping file contains **6,169 rows and 6,169 unique GP practice codes**, confirming one row per practice.

The hierarchy represented is:

`GP practice → PCN → Sub-ICB location → ICB → NHS commissioning region`

The mapping contains:

- **1,299** PCN codes
- **106** Sub-ICB location codes
- **42** ICB codes
- **7** NHS commissioning region codes

No rows are missing those four hierarchy codes.

Comparing the mapping snapshot with the latest-submission file:

- all **6,124** practices in the latest-submission file are present in the mapping
- the mapping contains an additional **45** practice codes

As with the June notebook, this establishes a coverage difference but does not by itself explain why those additional practice codes are present.

## Summary Workbook

The March release also includes an Excel summary workbook.

This is different from the tidy CSV outputs: it is organised as a set of presentation/reporting tables rather than one analysis-ready rectangular dataset.

For now it is enough to record the workbook structure rather than trying to treat every sheet as another raw table.

In [21]:
summary_xls.sheet_names

['Title sheet',
 'Notes and definitions',
 'Table 1',
 'Table 2',
 'Table 3a',
 'Table 3b',
 'Table 3c',
 'Table 4',
 'Table 5']

The workbook contains:

- a title sheet
- notes and definitions
- Tables 1, 2, 3a, 3b, 3c, 4 and 5

If the Atlas later needs to reproduce a published headline statistic, these tables may be useful as a cross-check. They should not be the preferred source for the core data model while the underlying CSV measures are available.

## What this structure pass has established

The March 2026 release uses the **older, more fragmented publication structure**.

The important points for later modelling are:

- most analytical files contain a **13-month rolling time series**
- diagnosis-rate data are separated into NHS and local-authority geography files
- demographic dementia-register breakdowns are split across several separate Sub-ICB files
- in those older files, categories such as age/sex, ethnicity, dementia type and residential type are encoded in the `Measure` value itself
- other measure files repeat observations across Sub-ICB, ICB, region and country levels
- the practice latest-submission file and mapping file are supporting datasets rather than measure tables
- the mapping file includes PCNs and gives a clean practice-to-commissioning hierarchy
- some historical months have different row counts or measure availability, so transformations should not assume that the entire 13-month series has a perfectly fixed schema
- the Excel summary workbook is a reporting artefact rather than the best basis for the Atlas data model

This explains why the March raw folder looks much larger and more complicated than June.

For the Atlas, a sensible next step would be to create a **March → June measure mapping** so the old split-file structure can be normalised into the newer consolidated concepts without losing the historical time series.